# Patient Management System

In this project, I created the file-based database MongoDB to create and host a database locally on my PC.This project shows my competency of managing patient records, appointments, and basic health information using MongoDB. Firstly, MongoDB compass was firstly installed from https://www.mongodb.com/try/download/shell and then set up the connection. The final output on MongoDB compass after finishing running this project was also attached as the screenshot as the same directory of this notebook. 


In [2]:
!pip install pymongo


   ---------------------------------------- 0.0/907.6 kB ? eta -:--:--
   ---------------------------------------- 907.6/907.6 kB 6.9 MB/s eta 0:00:00


In [9]:
!apt-get update
!apt-get install -y mongodb

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,928 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,835 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [

In [10]:
!service mongodb start

mongodb: unrecognized service


In [3]:
from pymongo import MongoClient
import datetime
import pprint

In [4]:
client = MongoClient("mongodb://localhost:27017/")
db = client["patient_db"] 
patients_collection = db["patients"] 

---
## 1: Add a Patient Record:

Creates a new document in the patients collection with the following information.

- `patient_id` (integer) - A simple unique identifier.
- `first_name` (string)
- `last_name` (string)
- `date_of_birth` (date)


In [7]:
def add_patient(patient_id, first_name, last_name, date_of_birth):
    patient = {
        "patient_id": patient_id,
        "first_name": first_name,
        "last_name": last_name,
        "date_of_birth": date_of_birth
    }

    patients_collection.insert_one(patient)
    print("Patient added successfully")


In [11]:
add_patient(1, "John", "Doe", "1990-05-14")
add_patient(2, "Alice", "Smith", "1988-11-02")
add_patient(3, "Michael", "Johnson", "1992-07-19")

Patient added successfully


### Discussion 1.1




In MongoDB, the _id field is automatically generated for every document and uniquely identifies it within the collection, usually using an ObjectId that is long and not human-readable. In contrast, patient_id is a custom identifier that we manually add to represent the patient within the application. Even though _id already guarantees uniqueness, a separate patient_id is useful because it is easier for humans to read, can follow a hospital’s existing numbering system, and can be used in application logic or when integrating with other healthcare systems.

---
## 2: List All Patients:

Queries the database to retrieve all patient records.
Displays a list of patients, showing their ID, name (first and last), and date of birth.


In [12]:
def list_patients():
    patients = patients_collection.find()

    for patient in patients:
        print(
            f"ID: {patient['patient_id']}, "
            f"Name: {patient['first_name']} {patient['last_name']}, "
            f"DOB: {patient['date_of_birth']}"
        )

In [13]:
list_patients()

ID: 1, Name: John Doe, DOB: 1990-05-14
ID: 2, Name: Alice Smith, DOB: 1988-11-02
ID: 3, Name: Michael Johnson, DOB: 1992-07-19


### Discussion 2.1


In MongoDB, the find() method returns a cursor, which means it provides an iterator that retrieves documents one at a time as you loop through them, rather than loading all results into memory at once. In practice, this allows the program to process records sequentially without storing the entire dataset in a list. This is important when a collection contains millions of records, because loading everything into memory could be very slow or exceed memory limits, while a cursor allows efficient and scalable data retrieval.

---
## 3: Find a Patient by ID:


- Queries the database to find the patient with matching patient_id.
- Displays the full information of the found patient, or indicates if no patient with that ID was found.

In [16]:
def find_patient_by_id(patient_id):
    patient = patients_collection.find_one({"patient_id": patient_id})

    if patient:
        print("Patient found:")
        print(f"ID: {patient['patient_id']}")
        print(f"First Name: {patient['first_name']}")
        print(f"Last Name: {patient['last_name']}")
        print(f"Date of Birth: {patient['date_of_birth']}")
    else:
        print("No patient found with that ID.")

In [17]:
find_patient_by_id(1)

Patient found:
ID: 1
First Name: John
Last Name: Doe
Date of Birth: 1990-05-14


In [18]:
find_patient_by_id(4)

No patient found with that ID.


### Discussion 3.1




If two documents had the same patient_id, the find_one() query in MongoDB would return only the first matching document it encounters, while the other document with the same ID would be ignored by that query. To prevent duplicate IDs from being inserted, we could create a unique index on the patient_id field so that the database automatically rejects any document that tries to use an existing patient_id which ensures that every patient ID remains unique.

---
## 4: Update Patient's Last Name:
- Finds the patient with matching `patient_id`.
- Updates the patient's `last_name` field with the provided new value.
- Informs the user if the update was successful or not.


In [19]:
def update_patient_last_name(patient_id, new_last_name):
    result = patients_collection.update_one(
        {"patient_id": patient_id},
        {"$set": {"last_name": new_last_name}}
    )

    if result.matched_count > 0:
        print("Patient last name updated successfully.")
    else:
        print("No patient found with that ID.")

In [20]:
find_patient_by_id(1)

Patient found:
ID: 1
First Name: John
Last Name: Doe
Date of Birth: 1990-05-14


In [21]:
update_patient_last_name(1, "Smith")

Patient last name updated successfully.


In [22]:
find_patient_by_id(1)

Patient found:
ID: 1
First Name: John
Last Name: Smith
Date of Birth: 1990-05-14


### Discussion 4.1


In MongoDB, using update_many() would be a bug in this scenario because the patient_id is intended to uniquely identify a single patient. If multiple documents accidentally had the same patient_id, update_many() would update the last name for all of them, potentially modifying multiple patient records incorrectly. Since the operation is meant to update only one specific patient, update_one() is the correct choice to ensure that only a single record is modified.

In [23]:
list_patients()

ID: 1, Name: John Smith, DOB: 1990-05-14
ID: 2, Name: Alice Smith, DOB: 1988-11-02
ID: 3, Name: Michael Johnson, DOB: 1992-07-19


---
## 5: Find Patients Born After a Given Year:

- Queries the database to find all patients who were born after the given year.
- Displays a list of these patients.


In [24]:
def find_patients_born_after_year(year):
    cutoff = f"{year}-01-01"

    patients = patients_collection.find(
        {"date_of_birth": {"$gt": cutoff}}
    )

    found = False
    for patient in patients:
        found = True
        print(
            f"ID: {patient['patient_id']}, "
            f"Name: {patient['first_name']} {patient['last_name']}, "
            f"DOB: {patient['date_of_birth']}"
        )

    if not found:
        print("No patients found born after that year.")

In [25]:
find_patients_born_after_year(1989)

ID: 1, Name: John Smith, DOB: 1990-05-14
ID: 3, Name: Michael Johnson, DOB: 1992-07-19


### Discussion 5.1




---
## 6: Add Contact Information:

Updates the document of the specified patient_id to include the email and phone number.


In [29]:
def find_patient_by_id(patient_id):
    patient = patients_collection.find_one({"patient_id": patient_id})

    if patient:
        print("Patient found:")
        print("ID:", patient["patient_id"])
        print("First Name:", patient["first_name"])
        print("Last Name:", patient["last_name"])
        print("Date of Birth:", patient["date_of_birth"])

        if "email" in patient:
            print("Email:", patient["email"])
        if "phone" in patient:
            print("Phone:", patient["phone"])

    else:
        print("No patient found with that ID.")

In [30]:
add_patient_contact(1, "john@example.com", "123-456-7890")

Contact information added successfully.


In [31]:
find_patient_by_id(1)

Patient found:
ID: 1
First Name: John
Last Name: Smith
Date of Birth: 1990-05-14
Email: john@example.com
Phone: 123-456-7890


### Discussion 6.1



---
## 7: Delete a Patient Record:

Removes the patient with matching patient_id from the patients collection.
Informs the user about successful or unsuccessful deletion.


In [32]:
def delete_patient_by_id(patient_id):
    result = patients_collection.delete_one({"patient_id": patient_id})

    if result.deleted_count > 0:
        print("Patient record deleted successfully.")
    else:
        print("No patient found with that ID.")

In [33]:
delete_patient_by_id(2)

Patient record deleted successfully.


In [34]:
find_patient_by_id(2)

No patient found with that ID.


### Quest 7.1

What would happen if you used `delete_many()` with the same query? In a real system, is permanently deleting a patient record ever a good idea, or can you think of a better approach?

---
## 8: Bring it together

In [35]:
from datetime import datetime

def main():
    print("\nOptions:")
    print("1. Add a patient")
    print("2. List all patients")
    print("3. Find a patient by ID")
    print("4. Update patient's last name")
    print("5. Find patients born after a given year")
    print("6. Add patient contact info")
    print("7. Delete a patient")

    choice = input("Enter your choice: ")

    if choice == "1":
        patient_id = int(input("Enter patient ID: "))
        first_name = input("Enter first name: ")
        last_name = input("Enter last name: ")
        dob_str = input("Enter date of birth (YYYY-MM-DD): ")
        dob = datetime.strptime(dob_str, "%Y-%m-%d")
        add_patient(patient_id, first_name, last_name, dob)

    elif choice == "2":
        list_patients()

    elif choice == "3":
        patient_id = int(input("Enter patient ID: "))
        find_patient_by_id(patient_id)

    elif choice == "4":
        patient_id = int(input("Enter patient ID: "))
        new_last_name = input("Enter new last name: ")
        update_patient_last_name(patient_id, new_last_name)

    elif choice == "5":
        year = int(input("Enter year: "))
        find_patients_born_after_year(year)

    elif choice == "6":
        patient_id = int(input("Enter patient ID: "))
        email = input("Enter email: ")
        phone = input("Enter phone number: ")
        add_patient_contact(patient_id, email, phone)

    elif choice == "7":
        patient_id = int(input("Enter patient ID: "))
        delete_patient_by_id(patient_id)

    else:
        print("Invalid choice.")


if __name__ == "__main__":
    main()


Options:
1. Add a patient
2. List all patients
3. Find a patient by ID
4. Update patient's last name
5. Find patients born after a given year
6. Add patient contact info
7. Delete a patient


Enter your choice:  2


ID: 1, Name: John Smith, DOB: 1990-05-14
ID: 3, Name: Michael Johnson, DOB: 1992-07-19


---
## Finished? Don't forget

Deliverables:
- A Jupyter Notebook: insert your codes and answers in the code skeleton HospitalDB.ipynb
- Your database, collection, and documents (a screen shot/s of MongoDB Compass is enough)

Grading Criteria:
- Correctness: Does the code correctly implement all required functionalities?
- Code Readability: Is the code organized, easy to read, and commented?
- Error Handling: Are basic error cases handled (e.g., patient not found)?
- Are pymongo and MongoDB used effectively?
